# Handling Missing Data and Ragged Edges

**Goal:** Master Kalman filtering with missing observations for real-time nowcasting with ragged edges.

**What you'll learn:**
- Simulate realistic publication lag patterns
- Implement Kalman filter with arbitrary missing data
- Compare to naive imputation (forward-fill, interpolation)
- Quantify value of timely indicators
- Handle completely missing quarters (GDP nowcasting)

**Time:** 12-15 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import inv
from statsmodels.tsa.statespace.dynamic_factor import DynamicFactor
import warnings
warnings.filterwarnings('ignore')

print("Missing Data Toolkit Loaded")

## Quick Win: Kalman Filter with Missing Data in 3 Minutes

In [ ]:
# Generate complete data
np.random.seed(42)
T, N, r = 200, 5, 2

# True factors (VAR(1))
Phi = np.array([[0.7, 0.1], [0.2, 0.6]])
Q = np.eye(r)
F_true = np.zeros((T, r))
for t in range(1, T):
    F_true[t] = Phi @ F_true[t-1] + np.random.multivariate_normal(np.zeros(r), Q)

# Loadings and data
Lambda = np.random.randn(N, r)
Lambda[:3, 0] = np.abs(Lambda[:3, 0]) * 1.5  # First 3 load on factor 1
Lambda[3:, 1] = np.abs(Lambda[3:, 1]) * 1.5  # Last 2 load on factor 2

Sigma_e = 0.5 * np.eye(N)
e = np.random.multivariate_normal(np.zeros(N), Sigma_e, T)
X_complete = F_true @ Lambda.T + e

print(f"✓ Generated {T} periods of {N} variables with {r} latent factors")
print(f"Factor persistence (eigenvalues of Phi): {np.linalg.eigvals(Phi)}")

In [ ]:
# Impose RAGGED EDGE: last 10 periods, 60% of data missing
X_ragged = X_complete.copy()
missing_mask = np.random.rand(10, N) < 0.6
X_ragged[-10:][missing_mask] = np.nan

print("Missing data pattern (last 10 periods):")
print(pd.DataFrame(X_ragged[-10:]).isnull().astype(int))
print("\n(1 = missing, 0 = observed)")

missing_pct = np.isnan(X_ragged[-10:]).sum() / (10 * N) * 100
print(f"\n{missing_pct:.1f}% missing in ragged-edge period")

In [ ]:
def kalman_filter_missing(X, Lambda, Phi, Sigma_e, Q, F0, P0):
    """
    Kalman filter handling arbitrary missing data (NaNs).
    """
    T, N = X.shape
    r = Lambda.shape[1]
    
    F_filt = np.zeros((T, r))
    F_pred = np.zeros((T, r))
    P_filt = np.zeros((T, r, r))
    P_pred = np.zeros((T, r, r))
    
    F_prev = F0
    P_prev = P0
    
    for t in range(T):
        # === PREDICTION ===
        F_pred[t] = Phi @ F_prev
        P_pred[t] = Phi @ P_prev @ Phi.T + Q
        
        # === UPDATE (with missing data) ===
        obs_mask = ~np.isnan(X[t])
        obs_idx = np.where(obs_mask)[0]
        
        if len(obs_idx) == 0:
            # All missing: skip update
            F_filt[t] = F_pred[t]
            P_filt[t] = P_pred[t]
        else:
            # Subset to observed
            X_obs = X[t, obs_idx]
            Lambda_obs = Lambda[obs_idx, :]
            Sigma_obs = Sigma_e[np.ix_(obs_idx, obs_idx)]
            
            # Innovation
            v = X_obs - Lambda_obs @ F_pred[t]
            S = Lambda_obs @ P_pred[t] @ Lambda_obs.T + Sigma_obs
            
            # Kalman gain
            K = P_pred[t] @ Lambda_obs.T @ inv(S)
            
            # Update
            F_filt[t] = F_pred[t] + K @ v
            P_filt[t] = (np.eye(r) - K @ Lambda_obs) @ P_pred[t]
        
        F_prev = F_filt[t]
        P_prev = P_filt[t]
    
    return F_filt, F_pred, P_filt, P_pred

# Run Kalman filter on ragged data
print("\nRunning Kalman filter with missing data...")
F_filt, F_pred, P_filt, P_pred = kalman_filter_missing(
    X=X_ragged,
    Lambda=Lambda,
    Phi=Phi,
    Sigma_e=Sigma_e,
    Q=Q,
    F0=np.zeros(r),
    P0=10 * np.eye(r)
)

# Factor estimation error
factor_mse = np.mean((F_filt - F_true)**2)
ragged_mse = np.mean((F_filt[-10:] - F_true[-10:])**2)

print(f"✓ Kalman filter completed")
print(f"\nFactor estimation MSE:")
print(f"  Full sample: {factor_mse:.4f}")
print(f"  Ragged edge (last 10): {ragged_mse:.4f}")
print(f"  → Error {ragged_mse/factor_mse:.1f}x higher during ragged edge")

## Comparison: Kalman Filter vs Naive Imputation

In [ ]:
# Method 1: Forward-fill (carry last observation forward)
X_ffill = pd.DataFrame(X_ragged).fillna(method='ffill').values

# Method 2: Linear interpolation
X_interp = pd.DataFrame(X_ragged).interpolate(method='linear').values

# Method 3: Zero-fill (assume missing = 0)
X_zero = np.nan_to_num(X_ragged, nan=0.0)

# Evaluate imputation quality (on true values)
missing_mask_full = np.isnan(X_ragged)

errors = {
    'Forward Fill': np.sqrt(np.mean((X_ffill[missing_mask_full] - X_complete[missing_mask_full])**2)),
    'Interpolation': np.sqrt(np.mean((X_interp[missing_mask_full] - X_complete[missing_mask_full])**2)),
    'Zero Fill': np.sqrt(np.mean((X_zero[missing_mask_full] - X_complete[missing_mask_full])**2)),
    'Kalman (via factors)': np.sqrt(np.mean((F_filt @ Lambda.T)[missing_mask_full] - X_complete[missing_mask_full])**2)
}

print("\nImputation RMSE (lower is better):")
print("="*50)
for method, rmse in sorted(errors.items(), key=lambda x: x[1]):
    print(f"  {method:20s}: {rmse:.4f}")
print("="*50)

best = min(errors, key=errors.get)
print(f"\n🏆 Best method: {best}")

## Visualize Factor Uncertainty During Ragged Edge

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Focus on last 30 periods
window = slice(-30, None)
t_idx = np.arange(T)[window]

for i in range(r):
    # True factor
    axes[i].plot(t_idx, F_true[window, i], 'k-', linewidth=2, label='True Factor', alpha=0.8)
    
    # Filtered estimate
    axes[i].plot(t_idx, F_filt[window, i], 'b--', linewidth=1.5, label='Kalman Estimate')
    
    # Uncertainty band (±2 std)
    std_filt = np.sqrt(P_filt[window, i, i])
    axes[i].fill_between(
        t_idx,
        F_filt[window, i] - 2*std_filt,
        F_filt[window, i] + 2*std_filt,
        alpha=0.3, color='blue', label='95% CI'
    )
    
    # Mark ragged edge region
    axes[i].axvspan(T-10, T, alpha=0.15, color='red', label='Ragged Edge' if i==0 else '')
    
    axes[i].set_ylabel(f'Factor {i+1}', fontsize=11)
    axes[i].legend(loc='upper left', fontsize=9)
    axes[i].grid(alpha=0.3)

axes[0].set_title('Kalman Filter: Factor Estimates with Ragged Edge', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Time Period', fontsize=11)
plt.tight_layout()
plt.show()

print("\nObservation:")
print("  - Uncertainty (blue band) WIDENS during ragged edge (red region)")
print("  - But factor estimates still track truth reasonably well!")
print("  - Kalman filter 'borrows strength' from observed variables")

## Realistic Publication Lags: GDP Nowcasting Scenario

In [ ]:
def impose_publication_lags(data_df, lag_dict):
    """
    Impose realistic publication lags.
    
    lag_dict: {column: days_delay}
    Converts days to months (approx)
    """
    data_ragged = data_df.copy()
    
    for col, days in lag_dict.items():
        months_lag = max(0, days // 30)
        if months_lag > 0 and col in data_ragged.columns:
            # Set last `months_lag` to NaN
            data_ragged.iloc[-months_lag:, data_ragged.columns.get_loc(col)] = np.nan
    
    return data_ragged

# Simulate monthly data (200 months)
dates = pd.date_range('2005-01-01', periods=200, freq='M')
monthly_df = pd.DataFrame(X_complete[:200], 
                         columns=['IP', 'Employment', 'Sales', 'PMI', 'Housing'],
                         index=dates)

# Realistic publication lags (as of end of month)
pub_lags = {
    'PMI': 1,           # Released within days
    'Employment': 5,    # Released ~5 days into next month
    'IP': 15,           # Released mid-next-month
    'Sales': 12,        # Released mid-next-month
    'Housing': 18       # Released later
}

monthly_ragged = impose_publication_lags(monthly_df, pub_lags)

print("Publication lag structure:")
for var, lag in pub_lags.items():
    print(f"  {var:12s}: {lag:2d} days")

print("\nMissing data pattern (last 3 months):")
print(monthly_ragged.tail(3).isnull().astype(int))

In [ ]:
# Use statsmodels DynamicFactor (handles missing data automatically)
print("\nEstimating DFM with publication lags...")

dfm_ragged = DynamicFactor(
    monthly_ragged,
    k_factors=2,
    factor_order=1,
    error_cov_type='diagonal'
)

dfm_ragged_res = dfm_ragged.fit(maxiter=500, disp=False)

# Extract factors (statsmodels handles missing data internally)
factors_ragged = dfm_ragged_res.factors.filtered

print(f"✓ DFM estimated with ragged edge")
print(f"  Log-likelihood: {dfm_ragged_res.llf:.1f}")

# Compare to DFM on complete data
dfm_complete = DynamicFactor(monthly_df, k_factors=2, factor_order=1)
dfm_complete_res = dfm_complete.fit(maxiter=500, disp=False)
factors_complete = dfm_complete_res.factors.filtered

# Factor correlation (account for rotation)
from scipy.linalg import orthogonal_procrustes
R, _ = orthogonal_procrustes(factors_ragged.values, factors_complete.values)
factors_ragged_aligned = factors_ragged.values @ R

factor_corr = np.corrcoef(factors_ragged_aligned[:, 0], factors_complete.values[:, 0])[0, 1]
print(f"\nFactor correlation (ragged vs complete): {factor_corr:.3f}")
print("  → High correlation means ragged edge doesn't hurt much!")

## Modify This: Experiment with Missing Patterns

In [ ]:
# CUSTOMIZE THESE
MISSING_FRACTION = 0.6  # Fraction of data missing (try: 0.3, 0.6, 0.9)
RAGGED_PERIODS = 10     # Number of periods with ragged edge (try: 5, 10, 20)

# Impose custom missing pattern
X_custom = X_complete.copy()
missing_mask = np.random.rand(RAGGED_PERIODS, N) < MISSING_FRACTION
X_custom[-RAGGED_PERIODS:][missing_mask] = np.nan

# Run Kalman filter
F_custom, _, P_custom, _ = kalman_filter_missing(
    X_custom, Lambda, Phi, Sigma_e, Q, np.zeros(r), 10*np.eye(r)
)

# Evaluate
mse_custom = np.mean((F_custom[-RAGGED_PERIODS:] - F_true[-RAGGED_PERIODS:])**2)
avg_uncertainty = np.mean([P_custom[-RAGGED_PERIODS:, i, i] for i in range(r)])

print(f"\nCustom missing pattern:")
print(f"  Missing fraction: {MISSING_FRACTION:.0%}")
print(f"  Ragged periods: {RAGGED_PERIODS}")
print(f"\nResults:")
print(f"  Factor MSE: {mse_custom:.4f}")
print(f"  Avg uncertainty: {avg_uncertainty:.4f}")
print(f"\n→ More missing data = higher uncertainty + higher error")

## Go Deeper

**Extensions:**
1. **Smoothed estimates:** Implement Kalman smoother (uses future data to improve past estimates)
2. **EM algorithm:** Estimate DFM parameters jointly with missing data
3. **MNAR patterns:** Model missingness mechanism (e.g., series drops out during crises)
4. **Mixed frequency:** Combine monthly + quarterly data (next module)

**Production tips:**
- Monitor which variables are missing in real-time
- Track nowcast uncertainty over time (should decrease as data arrives)
- Build "data availability dashboard" showing publication schedule
- Quantify "value of information" for each indicator

**Challenge:**
Build a function that:
1. Takes current data with ragged edge
2. Produces nowcast + uncertainty
3. Reports which variables are currently missing
4. Shows how much uncertainty would decrease if missing variable arrived